# Feature Engineering for Fraud Detection

This notebook demonstrates advanced feature engineering for transaction risk scoring and fraud detection using cleaned datasets. Features include transaction velocity, user profile, device consistency, geolocation deviation, merchant frequency, and SQL export for modeling.

In [13]:
# Import libraries
import pandas as pd
import numpy as np
import os

## Load Cleaned Data
Load processed datasets from EDA for feature engineering.

In [14]:
# Load cleaned datasets
creditcard_path = '../data/processed/creditcard_clean.csv'
ieee_path = '../data/processed/ieee_clean.csv'
paysim_path = '../data/processed/paysim_clean.csv'
df_credit = pd.read_csv(creditcard_path)
df_ieee = pd.read_csv(ieee_path)
df_paysim = pd.read_csv(paysim_path)

## Transaction Velocity Features
Calculate transaction velocity features (e.g., transactions per user per hour, amount velocity).

In [15]:
# Example: Transaction velocity for CreditCard dataset
if 'Time' in df_credit.columns and 'Amount' in df_credit.columns:
    df_credit['hour'] = (df_credit['Time'] // 3600).astype(int)
    velocity = df_credit.groupby(['hour'])['Amount'].agg(['count', 'sum']).reset_index()
    print(velocity.head())

   hour  count        sum
0     0   3963  257101.87
1     1   2217  146105.69
2     2   1576  108819.17
3     3   1821   94306.84
4     4   1082   79840.62


## User Profile Features
Aggregate user-level features (e.g., mean transaction amount, transaction frequency, fraud rate per user).

In [16]:
# Example: User profile features for PaySim dataset
if 'customer' in df_paysim.columns and 'amount' in df_paysim.columns and 'isFraud' in df_paysim.columns:
    user_profile = df_paysim.groupby('customer').agg({
        'amount': ['mean', 'sum', 'count'],
        'isFraud': 'mean'
    }).reset_index()
    print(user_profile.head())

## Device Consistency Features
Check device consistency (e.g., number of unique devices per user, device change frequency).

In [17]:
# Example: Device consistency for IEEE dataset
if 'DeviceInfo' in df_ieee.columns and 'card1' in df_ieee.columns:
    device_consistency = df_ieee.groupby('card1')['DeviceInfo'].nunique().reset_index().rename(columns={'DeviceInfo': 'unique_devices'})
    print(device_consistency.head())

   card1  unique_devices
0   1000               1
1   1001               0
2   1004               3
3   1005               1
4   1006               3


## Geolocation Deviation Features
Calculate geolocation deviation (e.g., distance between consecutive transactions, location change frequency).

In [18]:
# Example: Geolocation deviation (requires latitude/longitude columns)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c
if 'lat' in df_paysim.columns and 'lon' in df_paysim.columns:
    df_paysim = df_paysim.sort_values(['customer', 'step'])
    df_paysim['lat_prev'] = df_paysim.groupby('customer')['lat'].shift(1)
    df_paysim['lon_prev'] = df_paysim.groupby('customer')['lon'].shift(1)
    df_paysim['geo_dist'] = haversine(df_paysim['lat_prev'], df_paysim['lon_prev'], df_paysim['lat'], df_paysim['lon'])
    print(df_paysim[['customer', 'geo_dist']].head())

## Merchant Frequency Features
Calculate merchant frequency (e.g., transactions per merchant, fraud rate per merchant).

In [19]:
# Example: Merchant frequency for PaySim dataset
if 'merchant' in df_paysim.columns and 'isFraud' in df_paysim.columns:
    merchant_freq = df_paysim.groupby('merchant').agg({
        'amount': ['count', 'mean', 'sum'],
        'isFraud': 'mean'
    }).reset_index()
    print(merchant_freq.head())

## Export Engineered Features
Save engineered features to processed files or SQL tables for modeling.

In [20]:
# Save features to CSV (or SQL)
os.makedirs('../data/features', exist_ok=True)
if 'velocity' in locals():
    velocity.to_csv('../data/features/creditcard_velocity.csv', index=False)
if 'user_profile' in locals():
    user_profile.to_csv('../data/features/paysim_user_profile.csv', index=False)
if 'device_consistency' in locals():
    device_consistency.to_csv('../data/features/ieee_device_consistency.csv', index=False)
if 'merchant_freq' in locals():
    merchant_freq.to_csv('../data/features/paysim_merchant_freq.csv', index=False)

## Next Steps
Proceed to model training and evaluation using engineered features.

## Advanced Feature Engineering
Add rolling statistics, anomaly scores, categorical encoding, feature selection, and SQL integration for production-grade modeling.

In [21]:
# Rolling statistics: transaction amount mean/std per user (last 5 transactions)
if 'customer' in df_paysim.columns and 'amount' in df_paysim.columns and 'step' in df_paysim.columns:
    df_paysim = df_paysim.sort_values(['customer', 'step'])
    df_paysim['amount_roll_mean'] = df_paysim.groupby('customer')['amount'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
    df_paysim['amount_roll_std'] = df_paysim.groupby('customer')['amount'].rolling(window=5, min_periods=1).std().reset_index(level=0, drop=True)
    print(df_paysim[['customer', 'amount', 'amount_roll_mean', 'amount_roll_std']].head(10))

In [22]:
# Anomaly score: Z-score for transaction amount per user
if 'amount' in df_paysim.columns and 'customer' in df_paysim.columns:
    df_paysim['amount_zscore'] = df_paysim.groupby('customer')['amount'].transform(lambda x: (x - x.mean()) / x.std())
    print(df_paysim[['customer', 'amount', 'amount_zscore']].head(10))

In [23]:
# Categorical encoding: one-hot for transaction type (PaySim)
if 'type' in df_paysim.columns:
    type_dummies = pd.get_dummies(df_paysim['type'], prefix='type')
    df_paysim = pd.concat([df_paysim, type_dummies], axis=1)
    print(df_paysim.filter(like='type_').head())

   type_CASH_IN  type_CASH_OUT  type_DEBIT  type_PAYMENT  type_TRANSFER
0         False          False       False          True          False
1         False          False       False          True          False
2         False          False       False         False           True
3         False           True       False         False          False
4         False          False       False          True          False


In [24]:
# Feature selection: remove highly correlated features (example for PaySim)
numeric_cols = df_paysim.select_dtypes(include=[np.number]).columns
corr_matrix = df_paysim[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df_paysim_selected = df_paysim.drop(columns=to_drop)
print('Dropped columns:', to_drop)

Dropped columns: ['newbalanceOrig', 'newbalanceDest']


In [25]:
# SQL integration: export features to SQLite for modeling
import sqlite3
conn = sqlite3.connect('../data/features/features.db')
df_paysim_selected.to_sql('paysim_features', conn, if_exists='replace', index=False)
conn.close()